# Mutual Fund Performance Analytics

## Bluestock Fintech Internship

### Objectives

- Calculate Daily Returns
- Compute CAGR (1Y, 3Y, 5Y)
- Calculate Sharpe Ratio
- Calculate Sortino Ratio
- Calculate Alpha & Beta
- Compute Maximum Drawdown
- Generate Fund Scorecard
- Benchmark Comparison

In [1]:
import pandas as pd
import numpy as np
import sqlite3

import plotly.express as px
import plotly.graph_objects as go

from scipy import stats

In [2]:
conn = sqlite3.connect("../bluestock_mf.db")

print("Database Connected Successfully!")

Database Connected Successfully!


In [3]:
tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)

tables

,name
0,sqlite_sequence
1,dim_fund
2,dim_date
3,fact_nav
4,fact_transactions
5,fact_performance
6,fact_aum


In [4]:
nav_data = pd.read_sql("""
SELECT *
FROM fact_nav
""", conn)

nav_data.head()

,nav_id,fund_id,date_id,nav
0,1,6,3,520.4608
1,2,6,4,515.0971
2,3,6,5,521.7239
3,4,6,6,515.7880
4,5,6,7,515.1639


In [5]:
nav_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   nav_id   46000 non-null  int64  
 1   fund_id  46000 non-null  int64  
 2   date_id  46000 non-null  int64  
 3   nav      46000 non-null  float64
dtypes: float64(1), int64(3)
memory usage: 1.4 MB


In [6]:
date_data = pd.read_sql_query(
    "SELECT * FROM dim_date",
    conn
)

date_data.head()

,date_id,full_date,day,month,month_name,quarter,year,weekday
0,1,2022-01-01 00:00:00.000000,1,1,January,1,2022,Saturday
1,2,2022-01-02 00:00:00.000000,2,1,January,1,2022,Sunday
2,3,2022-01-03 00:00:00.000000,3,1,January,1,2022,Monday
3,4,2022-01-04 00:00:00.000000,4,1,January,1,2022,Tuesday
4,5,2022-01-05 00:00:00.000000,5,1,January,1,2022,Wednesday


In [7]:
fund_data = pd.read_sql_query("""
SELECT * FROM dim_fund
""", conn)

fund_data.head()

,fund_id,amfi_code,scheme_name,fund_house,category,sub_category,plan,benchmark,fund_manager,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,risk_category,sebi_category_code
0,1,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Equity,Large Cap,Regular,NIFTY 100 TRI,Sohini Andani,1.54,1.0,500.0,1000.0,Moderate,EC01
1,2,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Equity,Large Cap,Direct,NIFTY 100 TRI,Sohini Andani,0.66,1.0,500.0,1000.0,Moderate,EC01
2,3,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Equity,Small Cap,Regular,BSE 250 SmallCap TRI,R. Srinivasan,1.43,1.0,500.0,1000.0,Very High,EC03
3,4,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Equity,Small Cap,Direct,BSE 250 SmallCap TRI,R. Srinivasan,0.72,1.0,500.0,1000.0,Very High,EC03
4,5,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Debt,Gilt,Regular,CRISIL Dynamic Gilt Index,Dinesh Ahuja,0.77,0.0,500.0,1000.0,Low,DC02


In [8]:
nav_analysis = (
    nav_data
    .merge(
        date_data[["date_id", "full_date"]],
        on="date_id",
        how="left"
    )
    .merge(
        fund_data[["fund_id", "scheme_name", "fund_house"]],
        on="fund_id",
        how="left"
    )
)

nav_analysis.head()

,nav_id,fund_id,date_id,nav,full_date,scheme_name,fund_house
0,1,6,3,520.4608,2022-01-03 00:00:00.000000,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund
1,2,6,4,515.0971,2022-01-04 00:00:00.000000,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund
2,3,6,5,521.7239,2022-01-05 00:00:00.000000,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund
3,4,6,6,515.7880,2022-01-06 00:00:00.000000,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund
4,5,6,7,515.1639,2022-01-07 00:00:00.000000,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund


In [9]:
nav_analysis["full_date"] = pd.to_datetime(nav_analysis["full_date"])

nav_analysis = nav_analysis.sort_values(
    ["fund_id", "full_date"]
)

nav_analysis.head()

,nav_id,fund_id,date_id,nav,full_date,scheme_name,fund_house
21850,21851,1,3,54.3856,2022-01-03,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund
21851,21852,1,4,54.3474,2022-01-04,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund
21852,21853,1,5,54.6869,2022-01-05,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund
21853,21854,1,6,55.4550,2022-01-06,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund
21854,21855,1,7,55.3692,2022-01-07,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund


In [10]:
nav_analysis["daily_return"] = (
    nav_analysis
    .groupby("fund_id")["nav"]
    .pct_change()
)

nav_analysis.head(10)

,nav_id,fund_id,date_id,nav,full_date,scheme_name,fund_house,daily_return
21850,21851,1,3,54.3856,2022-01-03,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,NaN
21851,21852,1,4,54.3474,2022-01-04,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,-0.000702
21852,21853,1,5,54.6869,2022-01-05,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,0.006247
21853,21854,1,6,55.4550,2022-01-06,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,0.014045
21854,21855,1,7,55.3692,2022-01-07,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,-0.001547
21855,21856,1,10,55.2835,2022-01-10,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,-0.001548
21856,21857,1,11,56.0878,2022-01-11,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,0.014549
21857,21858,1,12,56.4978,2022-01-12,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,0.007310
21858,21859,1,13,56.2934,2022-01-13,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,-0.003618
21859,21860,1,14,56.5926,2022-01-14,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,0.005315


In [11]:
fig = px.histogram(
    nav_analysis,
    x="daily_return",
    nbins=100,
    title="Distribution of Daily Returns"
)

fig.show()

In [12]:
fig.write_image("../reports/daily_return_distribution.png")
print("Daily Return Distribution chart saved successfully!")

Daily Return Distribution chart saved successfully!


In [13]:
cagr_data = nav_analysis.groupby(
    ["fund_id", "scheme_name"]
).agg(
    start_nav=("nav", "first"),
    end_nav=("nav", "last"),
    start_date=("full_date", "first"),
    end_date=("full_date", "last")
).reset_index()

cagr_data.head()

,fund_id,scheme_name,start_nav,end_nav,start_date,end_date
0,1,SBI Bluechip Fund - Regular Plan - Growth,54.3856,149.3216,2022-01-03,2026-05-29
1,2,SBI Bluechip Fund - Direct Plan - Growth,58.4174,137.7323,2022-01-03,2026-05-29
2,3,SBI Small Cap Fund - Regular Plan - Growth,89.8738,309.2050,2022-01-03,2026-05-29
3,4,SBI Small Cap Fund - Direct Plan - Growth,96.4565,105.4849,2022-01-03,2026-05-29
4,5,SBI Magnum Gilt Fund - Regular Plan - Growth,42.1391,54.2038,2022-01-03,2026-05-29


In [14]:
cagr_data["years"] = (
    cagr_data["end_date"] -
    cagr_data["start_date"]
).dt.days / 365.25

cagr_data.head()

,fund_id,scheme_name,start_nav,end_nav,start_date,end_date,years
0,1,SBI Bluechip Fund - Regular Plan - Growth,54.3856,149.3216,2022-01-03,2026-05-29,4.399726
1,2,SBI Bluechip Fund - Direct Plan - Growth,58.4174,137.7323,2022-01-03,2026-05-29,4.399726
2,3,SBI Small Cap Fund - Regular Plan - Growth,89.8738,309.2050,2022-01-03,2026-05-29,4.399726
3,4,SBI Small Cap Fund - Direct Plan - Growth,96.4565,105.4849,2022-01-03,2026-05-29,4.399726
4,5,SBI Magnum Gilt Fund - Regular Plan - Growth,42.1391,54.2038,2022-01-03,2026-05-29,4.399726


In [15]:
cagr_data["cagr"] = (
    (cagr_data["end_nav"] / cagr_data["start_nav"])
    ** (1 / cagr_data["years"])
) - 1

cagr_data.head()

,fund_id,scheme_name,start_nav,end_nav,start_date,end_date,years,cagr
0,1,SBI Bluechip Fund - Regular Plan - Growth,54.3856,149.3216,2022-01-03,2026-05-29,4.399726,0.258047
1,2,SBI Bluechip Fund - Direct Plan - Growth,58.4174,137.7323,2022-01-03,2026-05-29,4.399726,0.215242
2,3,SBI Small Cap Fund - Regular Plan - Growth,89.8738,309.2050,2022-01-03,2026-05-29,4.399726,0.324235
3,4,SBI Small Cap Fund - Direct Plan - Growth,96.4565,105.4849,2022-01-03,2026-05-29,4.399726,0.020545
4,5,SBI Magnum Gilt Fund - Regular Plan - Growth,42.1391,54.2038,2022-01-03,2026-05-29,4.399726,0.058894


In [16]:
cagr_data["cagr_percent"] = (
    cagr_data["cagr"] * 100
).round(2)

cagr_data.head()

,fund_id,scheme_name,start_nav,end_nav,start_date,end_date,years,cagr,cagr_percent
0,1,SBI Bluechip Fund - Regular Plan - Growth,54.3856,149.3216,2022-01-03,2026-05-29,4.399726,0.258047,25.80
1,2,SBI Bluechip Fund - Direct Plan - Growth,58.4174,137.7323,2022-01-03,2026-05-29,4.399726,0.215242,21.52
2,3,SBI Small Cap Fund - Regular Plan - Growth,89.8738,309.2050,2022-01-03,2026-05-29,4.399726,0.324235,32.42
3,4,SBI Small Cap Fund - Direct Plan - Growth,96.4565,105.4849,2022-01-03,2026-05-29,4.399726,0.020545,2.05
4,5,SBI Magnum Gilt Fund - Regular Plan - Growth,42.1391,54.2038,2022-01-03,2026-05-29,4.399726,0.058894,5.89


In [17]:
top_cagr = cagr_data.sort_values(
    by="cagr_percent",
    ascending=False
)

top_cagr.head(10)

,fund_id,scheme_name,start_nav,end_nav,start_date,end_date,years,cagr,cagr_percent
12,13,ICICI Pru Midcap Fund - Regular - Growth,135.8720,473.7640,2022-01-03,2026-05-29,4.399726,0.328274,32.83
2,3,SBI Small Cap Fund - Regular Plan - Growth,89.8738,309.2050,2022-01-03,2026-05-29,4.399726,0.324235,32.42
39,40,DSP Small Cap Fund - Regular - Growth,81.6814,279.7511,2022-01-03,2026-05-29,4.399726,0.322874,32.29
36,37,Mirae Asset Tax Saver Fund - Regular - Growth,28.8620,97.7435,2022-01-03,2026-05-29,4.399726,0.319495,31.95
34,35,Mirae Asset Large Cap Fund - Regular - Growth,70.2514,230.2708,2022-01-03,2026-05-29,4.399726,0.309741,30.97
22,23,Kotak Flexicap Fund - Regular - Growth,49.9131,163.2397,2022-01-03,2026-05-29,4.399726,0.309075,30.91
7,8,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,107.3758,342.0072,2022-01-03,2026-05-29,4.399726,0.301232,30.12
38,39,DSP Midcap Fund - Regular - Growth,78.4622,245.3651,2022-01-03,2026-05-29,4.399726,0.295811,29.58
26,27,Axis Midcap Fund - Regular - Growth,68.3023,203.8581,2022-01-03,2026-05-29,4.399726,0.282144,28.21
0,1,SBI Bluechip Fund - Regular Plan - Growth,54.3856,149.3216,2022-01-03,2026-05-29,4.399726,0.258047,25.80


In [18]:
fig = px.bar(
    top_cagr.head(10),
    x="cagr_percent",
    y="scheme_name",
    orientation="h",
    text="cagr_percent",
    color="cagr_percent",
    title="Top 10 Mutual Funds by CAGR (%)"
)

fig.update_layout(
    xaxis_title="CAGR (%)",
    yaxis_title="Fund Name",
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

In [19]:
fig.write_image("../reports/top10_cagr.png")
print("Top 10 CAGR chart saved successfully!")

Top 10 CAGR chart saved successfully!


In [20]:
rf = 0.065
daily_rf = rf / 252

print("Annual Risk Free Rate:", rf)
print("Daily Risk Free Rate:", daily_rf)

Annual Risk Free Rate: 0.065
Daily Risk Free Rate: 0.00025793650793650796


In [21]:
sharpe_df = (
    sbi.groupby("fund_id")
    .agg(
        mean_return=("daily_return", "mean"),
        std_return=("daily_return", "std")
    )
    .reset_index()
)

sharpe_df.head()

NameError: name 'sbi' is not defined

In [ ]:
nav_df.head()

NameError: name 'nav_df' is not defined

In [22]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("../data/processed/mutual_funds.db")

In [23]:
query = """
SELECT
    n.nav_id,
    n.fund_id,
    n.date_id,
    n.nav,
    d.full_date,
    f.scheme_name,
    f.fund_house
FROM fact_nav n
JOIN dim_date d
ON n.date_id = d.date_id
JOIN dim_fund f
ON n.fund_id = f.fund_id
"""

nav_df = pd.read_sql(query, conn)

nav_df.head()

DatabaseError: Execution failed on sql '
SELECT
    n.nav_id,
    n.fund_id,
    n.date_id,
    n.nav,
    d.full_date,
    f.scheme_name,
    f.fund_house
FROM fact_nav n
JOIN dim_date d
ON n.date_id = d.date_id
JOIN dim_fund f
ON n.fund_id = f.fund_id
': no such table: fact_nav

In [24]:
import os

print(os.getcwd())

c:\Users\PRABHASH\Desktop\Mutual_Fund_Analytics\notebooks


In [25]:
import os

print(os.listdir("../data/processed"))

['02_nav_history.csv', '07_scheme_performance.csv', '08_investor_transactions.csv', 'mutual_funds.db']


In [26]:
import sqlite3

conn = sqlite3.connect("../data/processed/mutual_funds.db")

cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

print(cursor.fetchall())

[]


In [27]:
conn = sqlite3.connect("../bluestock_mf.db")

In [28]:
import sqlite3

conn = sqlite3.connect("../bluestock_mf.db")

cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

print(cursor.fetchall())

[('sqlite_sequence',), ('dim_fund',), ('dim_date',), ('fact_nav',), ('fact_transactions',), ('fact_performance',), ('fact_aum',)]


In [29]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../bluestock_mf.db")

query = """
SELECT
    n.nav_id,
    n.fund_id,
    n.date_id,
    n.nav,
    d.full_date,
    f.scheme_name,
    f.fund_house
FROM fact_nav n
JOIN dim_date d
ON n.date_id = d.date_id
JOIN dim_fund f
ON n.fund_id = f.fund_id
"""

nav_df = pd.read_sql(query, conn)

nav_df.head()

,nav_id,fund_id,date_id,nav,full_date,scheme_name,fund_house
0,1,6,3,520.4608,2022-01-03 00:00:00.000000,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund
1,2,6,4,515.0971,2022-01-04 00:00:00.000000,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund
2,3,6,5,521.7239,2022-01-05 00:00:00.000000,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund
3,4,6,6,515.7880,2022-01-06 00:00:00.000000,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund
4,5,6,7,515.1639,2022-01-07 00:00:00.000000,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund


In [30]:
nav_df["full_date"] = pd.to_datetime(nav_df["full_date"])

nav_df = nav_df.sort_values(
    ["fund_id", "full_date"]
)

nav_df["daily_return"] = (
    nav_df.groupby("fund_id")["nav"].pct_change()
)

nav_df.head()

,nav_id,fund_id,date_id,nav,full_date,scheme_name,fund_house,daily_return
21850,21851,1,3,54.3856,2022-01-03,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,NaN
21851,21852,1,4,54.3474,2022-01-04,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,-0.000702
21852,21853,1,5,54.6869,2022-01-05,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,0.006247
21853,21854,1,6,55.4550,2022-01-06,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,0.014045
21854,21855,1,7,55.3692,2022-01-07,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,-0.001547


In [31]:
sharpe_df = (
    nav_df.groupby("fund_id")
    .agg(
        mean_return=("daily_return", "mean"),
        std_return=("daily_return", "std")
    )
    .reset_index()
)

sharpe_df.head()

,fund_id,mean_return,std_return
0,1,0.000917,0.008656
1,2,0.000785,0.008781
2,3,0.001201,0.015837
3,4,0.000201,0.015717
4,5,0.000222,0.002499


In [32]:
rf = 0.065
daily_rf = rf / 252

sharpe_df["sharpe_ratio"] = (
    (sharpe_df["mean_return"] - daily_rf)
    / sharpe_df["std_return"]
) * np.sqrt(252)

sharpe_df.head()

,fund_id,mean_return,std_return,sharpe_ratio
0,1,0.000917,0.008656,1.208267
1,2,0.000785,0.008781,0.953279
2,3,0.001201,0.015837,0.945308
3,4,0.000201,0.015717,-0.057187
4,5,0.000222,0.002499,-0.226575


In [33]:
top_sharpe = (
    sharpe_df
    .merge(
        fund_data[["fund_id", "scheme_name"]],
        on="fund_id",
        how="left"
    )
    .sort_values("sharpe_ratio", ascending=False)
)

top_sharpe.head(10)

,fund_id,mean_return,std_return,sharpe_ratio,scheme_name
34,35,0.001074,0.008941,1.448291,Mirae Asset Large Cap Fund - Regular - Growth
22,23,0.001082,0.010008,1.306744,Kotak Flexicap Fund - Regular - Growth
36,37,0.001124,0.011134,1.234930,Mirae Asset Tax Saver Fund - Regular - Growth
0,1,0.000917,0.008656,1.208267,SBI Bluechip Fund - Regular Plan - Growth
12,13,0.001161,0.012152,1.180101,ICICI Pru Midcap Fund - Regular - Growth
38,39,0.001055,0.011179,1.132122,DSP Midcap Fund - Regular - Growth
7,8,0.001080,0.011929,1.093699,HDFC Mid-Cap Opportunities Fund - Regular - Gr...
15,16,0.000865,0.008913,1.081659,Nippon India Large Cap Fund - Regular - Growth
28,29,0.000852,0.009177,1.027213,ABSL Frontline Equity Fund - Regular - Growth
11,12,0.000843,0.009048,1.026524,ICICI Pru Bluechip Fund - Direct - Growth


In [34]:
downside_df = nav_df.copy()

downside_df["downside_return"] = downside_df["daily_return"].where(
    downside_df["daily_return"] < 0,
    0
)

downside_df.head()

,nav_id,fund_id,date_id,nav,full_date,scheme_name,fund_house,daily_return,downside_return
21850,21851,1,3,54.3856,2022-01-03,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,NaN,0.000000
21851,21852,1,4,54.3474,2022-01-04,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,-0.000702,-0.000702
21852,21853,1,5,54.6869,2022-01-05,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,0.006247,0.000000
21853,21854,1,6,55.4550,2022-01-06,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,0.014045,0.000000
21854,21855,1,7,55.3692,2022-01-07,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,-0.001547,-0.001547


In [35]:
sortino_df = (
    downside_df.groupby("fund_id")
    .agg(
        mean_return=("daily_return", "mean"),
        downside_std=("downside_return", "std")
    )
    .reset_index()
)

sortino_df.head()

,fund_id,mean_return,downside_std
0,1,0.000917,0.004661
1,2,0.000785,0.004852
2,3,0.001201,0.008657
3,4,0.000201,0.009163
4,5,0.000222,0.001387


In [36]:
rf = 0.065
daily_rf = rf / 252

sortino_df["sortino_ratio"] = (
    (sortino_df["mean_return"] - daily_rf)
    / sortino_df["downside_std"]
) * np.sqrt(252)

sortino_df.head()

,fund_id,mean_return,downside_std,sortino_ratio
0,1,0.000917,0.004661,2.243897
1,2,0.000785,0.004852,1.725274
2,3,0.001201,0.008657,1.729395
3,4,0.000201,0.009163,-0.098093
4,5,0.000222,0.001387,-0.408287


In [37]:
fund_data = pd.read_sql("""
SELECT
    fund_id,
    scheme_name
FROM dim_fund
""", conn)

sortino_df = sortino_df.merge(
    fund_data,
    on="fund_id",
    how="left"
)

sortino_df.head()

,fund_id,mean_return,downside_std,sortino_ratio,scheme_name
0,1,0.000917,0.004661,2.243897,SBI Bluechip Fund - Regular Plan - Growth
1,2,0.000785,0.004852,1.725274,SBI Bluechip Fund - Direct Plan - Growth
2,3,0.001201,0.008657,1.729395,SBI Small Cap Fund - Regular Plan - Growth
3,4,0.000201,0.009163,-0.098093,SBI Small Cap Fund - Direct Plan - Growth
4,5,0.000222,0.001387,-0.408287,SBI Magnum Gilt Fund - Regular Plan - Growth


In [38]:
mdd_df = nav_df.copy()

mdd_df["running_max"] = (
    mdd_df.groupby("fund_id")["nav"]
    .cummax()
)

mdd_df.head()

,nav_id,fund_id,date_id,nav,full_date,scheme_name,fund_house,daily_return,running_max
21850,21851,1,3,54.3856,2022-01-03,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,NaN,54.3856
21851,21852,1,4,54.3474,2022-01-04,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,-0.000702,54.3856
21852,21853,1,5,54.6869,2022-01-05,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,0.006247,54.6869
21853,21854,1,6,55.4550,2022-01-06,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,0.014045,55.4550
21854,21855,1,7,55.3692,2022-01-07,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,-0.001547,55.4550


In [39]:
mdd_df["drawdown"] = (
    (mdd_df["nav"] - mdd_df["running_max"])
    / mdd_df["running_max"]
)

mdd_df.head()

,nav_id,fund_id,date_id,nav,full_date,scheme_name,fund_house,daily_return,running_max,drawdown
21850,21851,1,3,54.3856,2022-01-03,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,NaN,54.3856,0.000000
21851,21852,1,4,54.3474,2022-01-04,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,-0.000702,54.3856,-0.000702
21852,21853,1,5,54.6869,2022-01-05,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,0.006247,54.6869,0.000000
21853,21854,1,6,55.4550,2022-01-06,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,0.014045,55.4550,0.000000
21854,21855,1,7,55.3692,2022-01-07,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,-0.001547,55.4550,-0.001547


In [40]:
max_drawdown = (
    mdd_df.groupby("fund_id")
    .agg(
        max_drawdown=("drawdown", "min")
    )
    .reset_index()
)

max_drawdown.head()

,fund_id,max_drawdown
0,1,-0.150124
1,2,-0.118035
2,3,-0.287060
3,4,-0.525742
4,5,-0.043287


In [41]:
alpha_beta = pd.read_sql("""
SELECT
    f.scheme_name,
    p.alpha,
    p.beta
FROM fact_performance p
JOIN dim_fund f
ON p.fund_id = f.fund_id
""", conn)

alpha_beta.head()

,scheme_name,alpha,beta
0,SBI Bluechip Fund - Regular Plan - Growth,0.87,0.89
1,SBI Bluechip Fund - Direct Plan - Growth,1.78,0.87
2,SBI Small Cap Fund - Regular Plan - Growth,1.23,0.89
3,SBI Small Cap Fund - Direct Plan - Growth,1.13,1.04
4,SBI Magnum Gilt Fund - Regular Plan - Growth,1.60,0.22


In [43]:
scorecard = (
    cagr_data[["fund_id", "scheme_name", "cagr_percent"]]
    .merge(
        sharpe_df[["fund_id", "sharpe_ratio"]],
        on="fund_id"
    )
    .merge(
        sortino_df[["fund_id", "sortino_ratio"]],
        on="fund_id"
    )
    .merge(
        max_drawdown[["fund_id", "max_drawdown"]],
        on="fund_id"
    )
    .merge(
        alpha_beta,
        on="scheme_name"
    )
)

scorecard.head()

,fund_id,scheme_name,cagr_percent,sharpe_ratio,sortino_ratio,max_drawdown,alpha,beta
0,1,SBI Bluechip Fund - Regular Plan - Growth,25.80,1.208267,2.243897,-0.150124,0.87,0.89
1,2,SBI Bluechip Fund - Direct Plan - Growth,21.52,0.953279,1.725274,-0.118035,1.78,0.87
2,3,SBI Small Cap Fund - Regular Plan - Growth,32.42,0.945308,1.729395,-0.287060,1.23,0.89
3,4,SBI Small Cap Fund - Direct Plan - Growth,2.05,-0.057187,-0.098093,-0.525742,1.13,1.04
4,5,SBI Magnum Gilt Fund - Regular Plan - Growth,5.89,-0.226575,-0.408287,-0.043287,1.60,0.22


In [44]:
benchmark = pd.read_sql("""
SELECT
    f.scheme_name,
    p.return_3yr_pct,
    p.benchmark_3yr_pct
FROM fact_performance p
JOIN dim_fund f
ON p.fund_id = f.fund_id
""", conn)

benchmark.head()

,scheme_name,return_3yr_pct,benchmark_3yr_pct
0,SBI Bluechip Fund - Regular Plan - Growth,12.36,11.49
1,SBI Bluechip Fund - Direct Plan - Growth,11.30,9.52
2,SBI Small Cap Fund - Regular Plan - Growth,23.39,22.16
3,SBI Small Cap Fund - Direct Plan - Growth,23.14,22.01
4,SBI Magnum Gilt Fund - Regular Plan - Growth,6.07,4.47


In [45]:
benchmark["outperformance"] = (
    benchmark["return_3yr_pct"]
    - benchmark["benchmark_3yr_pct"]
)

benchmark.head()

,scheme_name,return_3yr_pct,benchmark_3yr_pct,outperformance
0,SBI Bluechip Fund - Regular Plan - Growth,12.36,11.49,0.87
1,SBI Bluechip Fund - Direct Plan - Growth,11.30,9.52,1.78
2,SBI Small Cap Fund - Regular Plan - Growth,23.39,22.16,1.23
3,SBI Small Cap Fund - Direct Plan - Growth,23.14,22.01,1.13
4,SBI Magnum Gilt Fund - Regular Plan - Growth,6.07,4.47,1.60


In [46]:
import plotly.express as px

top10_sharpe = (
    sharpe_df
    .sort_values("sharpe_ratio", ascending=False)
    .head(10)
)

fig = px.bar(
    top10_sharpe,
    x="sharpe_ratio",
    y="scheme_name",
    orientation="h",
    color="sharpe_ratio",
    title="Top 10 Mutual Funds by Sharpe Ratio",
    text=top10_sharpe["sharpe_ratio"].round(2)
)

fig.update_layout(
    yaxis_title="Fund Name",
    xaxis_title="Sharpe Ratio"
)

fig.write_image("../reports/top10_sharpe_ratio.png")

fig.show()

print("Top10 Sharpe Ratio chart saved successfully!")

ValueError: Value of 'y' is not the name of a column in 'data_frame'. Expected one of ['fund_id', 'mean_return', 'std_return', 'sharpe_ratio'] but received: scheme_name

In [47]:
globals().keys()

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__vsc_ipynb_file__', '_i', '_ii', '_iii', '_i1', 'pd', 'np', 'sqlite3', 'px', 'go', 'stats', '_i2', 'conn', '_i3', 'tables', '_3', '_i4', 'nav_data', '_4', '_i5', '_i6', 'date_data', '_6', '_i7', 'fund_data', '_7', '_i8', 'nav_analysis', '_8', '_i9', '_9', '_i10', '_10', '_i11', 'fig', '_i12', '_i13', 'cagr_data', '_13', '_i14', '_14', '_i15', '_15', '_i16', '_16', '_i17', 'top_cagr', '_17', '_i18', '_i19', '_i20', 'rf', 'daily_rf', '_i21', '_i22', '_i23', 'query', '_i24', 'os', '_i25', '_i26', 'cursor', '_i27', '_i28', '_i29', 'nav_df', '_29', '_i30', '_30', '_i31', 'sharpe_df', '_31', '_i32', '_32', '_i33', 'top_sharpe', '_33', '_i34', 'downside_df', '_34', '_i35', 'sortino_df', '_35', '_i36', '_36', '_i37', '_37', '_i38', 'mdd_df', '_38', '_i39', '_39', '_i40', 'max_drawdown', '_40', '_i4

In [48]:
sharpe_df = sharpe_df.merge(
    fund_data[["fund_id", "scheme_name"]],
    on="fund_id",
    how="left"
)

sharpe_df.head()

,fund_id,mean_return,std_return,sharpe_ratio,scheme_name
0,1,0.000917,0.008656,1.208267,SBI Bluechip Fund - Regular Plan - Growth
1,2,0.000785,0.008781,0.953279,SBI Bluechip Fund - Direct Plan - Growth
2,3,0.001201,0.015837,0.945308,SBI Small Cap Fund - Regular Plan - Growth
3,4,0.000201,0.015717,-0.057187,SBI Small Cap Fund - Direct Plan - Growth
4,5,0.000222,0.002499,-0.226575,SBI Magnum Gilt Fund - Regular Plan - Growth


In [49]:
top10_sharpe = (
    sharpe_df
    .sort_values("sharpe_ratio", ascending=False)
    .head(10)
)

fig = px.bar(
    top10_sharpe,
    x="sharpe_ratio",
    y="scheme_name",
    orientation="h",
    color="sharpe_ratio",
    text="sharpe_ratio",
    title="Top 10 Mutual Funds by Sharpe Ratio"
)

fig.update_traces(texttemplate="%{text:.2f}")

fig.write_image("../reports/top10_sharpe_ratio.png")

fig.show()

print("Top10 Sharpe Ratio chart saved successfully!")

Top10 Sharpe Ratio chart saved successfully!


In [50]:
sortino_df = sortino_df.merge(
    fund_data[["fund_id", "scheme_name"]],
    on="fund_id",
    how="left"
)

top10_sortino = (
    sortino_df
    .sort_values("sortino_ratio", ascending=False)
    .head(10)
)

fig = px.bar(
    top10_sortino,
    x="sortino_ratio",
    y="scheme_name",
    orientation="h",
    color="sortino_ratio",
    text="sortino_ratio",
    title="Top 10 Mutual Funds by Sortino Ratio"
)

fig.update_traces(texttemplate="%{text:.2f}")

fig.write_image("../reports/top10_sortino_ratio.png")

fig.show()

print("✅ Top10 Sortino Ratio chart saved successfully!")

ValueError: Value of 'y' is not the name of a column in 'data_frame'. Expected one of ['fund_id', 'mean_return', 'downside_std', 'sortino_ratio', 'scheme_name_x', 'scheme_name_y'] but received: scheme_name

In [51]:
sortino_df.columns

Index(['fund_id', 'mean_return', 'downside_std', 'sortino_ratio',
       'scheme_name_x', 'scheme_name_y'],
      dtype='str')

In [52]:
top10_sortino = (
    sortino_df
    .sort_values("sortino_ratio", ascending=False)
    .head(10)
)

fig = px.bar(
    top10_sortino,
    x="sortino_ratio",
    y="scheme_name_x",   # <-- yahan change kiya hai
    orientation="h",
    color="sortino_ratio",
    text="sortino_ratio",
    title="Top 10 Mutual Funds by Sortino Ratio"
)

fig.update_traces(texttemplate="%{text:.2f}")

fig.write_image("../reports/top10_sortino_ratio.png")

fig.show()

print("Top10 Sortino Ratio chart saved successfully!")

Top10 Sortino Ratio chart saved successfully!


In [53]:
print(max_drawdown.columns)

Index(['fund_id', 'max_drawdown'], dtype='str')


In [54]:
max_drawdown = max_drawdown.merge(
    fund_data[["fund_id", "scheme_name"]],
    on="fund_id",
    how="left"
)

max_drawdown.head()

,fund_id,max_drawdown,scheme_name
0,1,-0.150124,SBI Bluechip Fund - Regular Plan - Growth
1,2,-0.118035,SBI Bluechip Fund - Direct Plan - Growth
2,3,-0.287060,SBI Small Cap Fund - Regular Plan - Growth
3,4,-0.525742,SBI Small Cap Fund - Direct Plan - Growth
4,5,-0.043287,SBI Magnum Gilt Fund - Regular Plan - Growth


In [55]:
top10_mdd = max_drawdown.sort_values(
    "max_drawdown",
    ascending=False
).head(10)

fig = px.bar(
    top10_mdd,
    x="max_drawdown",
    y="scheme_name",
    orientation="h",
    color="max_drawdown",
    text="max_drawdown",
    title="Top 10 Funds by Lowest Maximum Drawdown"
)

fig.update_traces(texttemplate="%{text:.2%}")

fig.write_image("../reports/top10_max_drawdown.png")

fig.show()

print("✅ Maximum Drawdown chart saved successfully!")

✅ Maximum Drawdown chart saved successfully!


In [56]:
top_benchmark = benchmark.sort_values(
    "outperformance",
    ascending=False
).head(10)

fig = px.bar(
    top_benchmark,
    x="outperformance",
    y="scheme_name",
    orientation="h",
    color="outperformance",
    text="outperformance",
    title="Top 10 Funds Outperforming Benchmark"
)

fig.update_traces(texttemplate="%{text:.2f}")

fig.write_image("../reports/benchmark_comparison.png")

fig.show()

print("✅ Benchmark Comparison chart saved successfully!")

✅ Benchmark Comparison chart saved successfully!


In [57]:
scorecard.to_csv(
    "../reports/fund_scorecard.csv",
    index=False
)

print("✅ Fund Scorecard exported successfully!")

✅ Fund Scorecard exported successfully!


In [58]:
import os

for file in sorted(os.listdir("../reports")):
    print(file)

age_distribution.png
aum_growth.png
benchmark_comparison.png
category_heatmap.png
city_tier_distribution.png
daily_return_distribution.png
data_quality_summary.txt
data_summary.txt
fund_scorecard.csv
gender_distribution.png
income_distribution.png
kyc_status.png
nav_trend.png
payment_mode.png
sip_trend.png
state_investment.png
top10_cagr.png
top10_max_drawdown.png
top10_sharpe_ratio.png
top10_sortino_ratio.png
